# Aula 18 - Grafos de Logística de Insumos, Processo e Veículos

Solução multicamada isolando de forma computacional as tubulações físicas da infraestrutura rodoviária civil da estação de serviço.

In [1]:
from dataclasses import dataclass
from heapq import heappop, heappush
from math import inf
from typing import Dict, List, Tuple

@dataclass(frozen=True)
class Rota:
    destino: str
    peso: float
    descricao: str

class GrafoLogistico:
    def __init__(self) -> None:
        self._adjacencias: Dict[str, List[Rota]] = {}

    def adicionar_rota(self, origem: str, destino: str, peso: float, descricao: str) -> None:
        if peso < 0:
            raise ValueError("Dijkstra exige pesos não negativos.")
        self._adjacencias.setdefault(origem, []).append(Rota(destino, peso, descricao))
        self._adjacencias.setdefault(destino, [])

    def menor_caminho(self, origem: str, destino: str) -> Tuple[float, List[str]]:
        distancias = {vertice: inf for vertice in self._adjacencias}
        predecessores: Dict[str, str] = {}
        distancias[origem] = 0.0
        fila: List[Tuple[float, str]] = [(0.0, origem)]

        while fila:
            distancia_atual, atual = heappop(fila)
            if distancia_atual != distancias[atual]:
                continue
            if atual == destino:
                break
            for rota in self._adjacencias[atual]:
                candidata = distancia_atual + rota.peso
                if candidata < distancias[rota.destino]:
                    distancias[rota.destino] = candidata
                    predecessores[rota.destino] = atual
                    heappush(fila, (candidata, rota.destino))

        if distancias[destino] == inf:
            raise ValueError(f"Não existe rota de {origem} para {destino}.")

        caminho = [destino]
        while caminho[-1] != origem:
            caminho.append(predecessores[caminho[-1]])
        caminho.reverse()
        return distancias[destino], caminho

def adicionar_rotas(grafo: GrafoLogistico, rotas: List[Tuple[str, str, float, str]]) -> None:
    for origem, destino, peso, descricao in rotas:
        grafo.adicionar_rota(origem, destino, peso, descricao)

fluxo_materiais = GrafoLogistico()
adicionar_rotas(fluxo_materiais, [
    ("Rede de Água", "Purificador OSM", 10, "entrada de água bruta"),
    ("Purificador OSM", "E-101", 15, "água deionizada para eletrólise"),
    ("E-101", "C-101", 5, "H2 bruto"),
    ("C-101", "TK-101", 10, "H2 armazenado a 350 bar"),
    ("C-101", "C-102", 15, "compressão de 2º estágio"),
    ("C-102", "TK-102", 8, "H2 armazenado a 700 bar"),
    ("TK-101", "MAN-101", 12, "linha de baixa pressao"),
    ("TK-102", "MAN-101", 15, "linha de alta pressao"),
    ("CH-101", "MAN-101", 6, "resfriamento criogênico a -40C (Protocolo J2601)"),
    ("MAN-101", "D-101", 4, "dosagem para o bico injetor"),
])

distancia_mat, caminho_mat = fluxo_materiais.menor_caminho("Rede de Água", "D-101")
print("=== REDE MULTICAMADA DA ESTAÇÃO H2 ===")
print("\n[Camada GM] Fluxo de Matéria Prima (Água até Dispenser de H2)")
print("  " + " -> ".join(caminho_mat))
print(f"  Custo Metragem Virtual: {distancia_mat:.0f} m")

circulacao_veiculos = GrafoLogistico()
anel_viario = [
    ("Entrada do Posto", "Fila de Espera", 30, "pista de acesso FCEV"),
    ("Fila de Espera", "Ilha de Abastecimento D-101", 20, "acesso à baia de recarga"),
    ("Fila de Espera", "Ilha de Abastecimento D-102", 25, "acesso à baia de recarga (secundária)"),
    ("Ilha de Abastecimento D-101", "Tóten de Pagamento", 15, "saída da baia"),
    ("Ilha de Abastecimento D-102", "Tóten de Pagamento", 10, "saída da baia"),
    ("Tóten de Pagamento", "Via Pública (Saída)", 25, "liberação de cancela"),
]
adicionar_rotas(circulacao_veiculos, anel_viario)

dist_veiculo, rota_veiculo = circulacao_veiculos.menor_caminho("Entrada do Posto", "Via Pública (Saída)")
print("\n[Camada GV] Circulação do Veículo (FCEV) na Planta")
print("  " + " -> ".join(rota_veiculo))
print(f"  Distância percorrida pelo cliente: {dist_veiculo:.0f} m")


=== REDE MULTICAMADA DA ESTAÇÃO H2 ===

[Camada GM] Fluxo de Matéria Prima (Água até Dispenser de H2)
  Rede de Água -> Purificador OSM -> E-101 -> C-101 -> TK-101 -> MAN-101 -> D-101
  Custo Metragem Virtual: 56 m

[Camada GV] Circulação do Veículo (FCEV) na Planta
  Entrada do Posto -> Fila de Espera -> Ilha de Abastecimento D-101 -> Tóten de Pagamento -> Via Pública (Saída)
  Distância percorrida pelo cliente: 90 m
